In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from scikeras.wrappers import KerasClassifier


In [5]:
print("--- 1. Data Exploration and Preprocessing ---")

url_or_path = "/content/sonardataset (3).csv"
try:
    # First, load assuming no header, then check if row 0 actually looks
    # like a header (i.e. contains non-numeric strings). This avoids the
    # "could not convert string to float" error when the CSV does have
    # column names in its first row.
    df_raw = pd.read_csv(url_or_path, header=None)
    first_row = df_raw.iloc[0]
    # Try to convert all but the last column (assumed target) to numeric
    looks_like_header = False
    for val in first_row[:-1]:
        try:
            float(val)
        except (ValueError, TypeError):
            looks_like_header = True
            break

    if looks_like_header:
        df = pd.read_csv(url_or_path, header=0)
    else:
        df = df_raw

    print(f"Dataset loaded successfully with shape: {df.shape}")
except FileNotFoundError:
    print("Please ensure 'sonardataset.csv' is in the same directory as this script.")
    exit()

# Identify the target column as the last column, regardless of whether
# it's named "60" (no-header case) or something else (header case).
target_col = df.columns[-1]

print(f"Number of samples: {df.shape[0]}")
print(f"Number of features: {df.shape[1] - 1}")
print("Class distribution:\n", df[target_col].value_counts())

# Handle missing values, if any
df.dropna(inplace=True)

# Separate features (X) and target (y)
# Explicit astype(float) guards against object-dtype arrays (e.g. if any
# stray non-numeric values slipped through) causing scaler errors later.
X = df.drop(columns=[target_col]).values.astype(float)
y = df[target_col].values

# Encode target labels ("M"/"R") into 0/1
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split (80/20), stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print("Data preprocessing complete. Data split into training and test sets.\n")


--- 1. Data Exploration and Preprocessing ---
Dataset loaded successfully with shape: (208, 61)
Number of samples: 208
Number of features: 60
Class distribution:
 Y
M    111
R     97
Name: count, dtype: int64
Data preprocessing complete. Data split into training and test sets.



In [6]:
print("--- 2. Basic Model Implementation ---")

def create_basic_model():
    model = Sequential([
        Dense(30, input_dim=60, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

base_model = create_basic_model()

print("Training the base model...")
history = base_model.fit(X_train, y_train, epochs=50, batch_size=16, verbose=0)

y_pred_base_probs = base_model.predict(X_test, verbose=0)
y_pred_base = (y_pred_base_probs > 0.5).astype(int).ravel()


--- 2. Basic Model Implementation ---
Training the base model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
import itertools

print("\n--- 3. Hyperparameter Tuning ---")
print("Running manual grid search over hyperparameter combinations...")



def create_tuned_model(neurons=15, learning_rate=0.001, activation='relu'):
    model = Sequential()
    model.add(Dense(neurons, input_dim=60, activation=activation))
    model.add(Dense(1, activation='sigmoid'))
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

param_grid = {
    'neurons': [15, 30, 60],
    'learning_rate': [0.01, 0.001],
    'activation': ['relu', 'tanh'],
    'batch_size': [16, 32],
    'epochs': [50, 100],
}


X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

keys = list(param_grid.keys())
combinations = list(itertools.product(*param_grid.values()))
print(f"Testing {len(combinations)} hyperparameter combinations...")

best_score = -1
best_params = None
best_model = None

for i, combo in enumerate(combinations):
    params = dict(zip(keys, combo))
    model = create_tuned_model(
        neurons=params['neurons'],
        learning_rate=params['learning_rate'],
        activation=params['activation'],
    )
    model.fit(
        X_tr, y_tr,
        epochs=params['epochs'],
        batch_size=params['batch_size'],
        verbose=0,
    )
    val_probs = model.predict(X_val, verbose=0)
    val_preds = (val_probs > 0.5).astype(int).ravel()
    score = accuracy_score(y_val, val_preds)

    if (i + 1) % 5 == 0 or i == len(combinations) - 1:
        print(f"  [{i + 1}/{len(combinations)}] params={params} -> val_accuracy={score:.4f}")

    if score > best_score:
        best_score = score
        best_params = params
        best_model = model

print(f"Best Validation Accuracy from Grid Search: {best_score:.4f}")
print(f"Best Hyperparameters: {best_params}")


final_model = create_tuned_model(
    neurons=best_params['neurons'],
    learning_rate=best_params['learning_rate'],
    activation=best_params['activation'],
)
final_model.fit(
    X_train, y_train,
    epochs=best_params['epochs'],
    batch_size=best_params['batch_size'],
    verbose=0,
)

y_pred_tuned_probs = final_model.predict(X_test, verbose=0)
y_pred_tuned = (y_pred_tuned_probs > 0.5).astype(int).ravel()



--- 3. Hyperparameter Tuning ---
Running manual grid search over hyperparameter combinations...
Testing 48 hyperparameter combinations...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr

  [5/48] params={'neurons': 15, 'learning_rate': 0.01, 'activation': 'tanh', 'batch_size': 16, 'epochs': 50} -> val_accuracy=0.7941


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr

  [10/48] params={'neurons': 15, 'learning_rate': 0.001, 'activation': 'relu', 'batch_size': 16, 'epochs': 100} -> val_accuracy=0.8235


...

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr

  [45/48] params={'neurons': 60, 'learning_rate': 0.001, 'activation': 'tanh', 'batch_size': 16, 'epochs': 50} -> val_accuracy=0.8235


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


  [48/48] params={'neurons': 60, 'learning_rate': 0.001, 'activation': 'tanh', 'batch_size': 32, 'epochs': 100} -> val_accuracy=0.8235
Best Validation Accuracy from Grid Search: 0.8824
Best Hyperparameters: {'neurons': 30, 'learning_rate': 0.01, 'activation': 'tanh', 'batch_size': 32, 'epochs': 50}


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
print("\n--- 4. Evaluation ---")

def evaluate_model(y_true, y_pred, model_name):
    print(f"\nEvaluation Metrics for {model_name}:")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1-Score:  {f1_score(y_true, y_pred):.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=encoder.classes_))

evaluate_model(y_test, y_pred_base, "Base Model (Default Params)")
evaluate_model(y_test, y_pred_tuned, "Tuned Model (Best Params)")

print("\n--- Discussion ---")
print("By comparing the metrics above, you can observe the effect of hyperparameter tuning.")
print("The Base Model uses a standard 30-neuron hidden layer and default Adam learning rate.")
print("The Tuned Model systematically tested different architectures and training lengths to optimize for accuracy.")
print("Generally, tuning improves Precision and Recall, reducing false positives (classifying a rock as a mine) or false negatives (missing a mine).")


--- 4. Evaluation ---

Evaluation Metrics for Base Model (Default Params):
Accuracy:  0.8571
Precision: 0.9375
Recall:    0.7500
F1-Score:  0.8333
Confusion Matrix:
[[21  1]
 [ 5 15]]

Classification Report:
              precision    recall  f1-score   support

           M       0.81      0.95      0.88        22
           R       0.94      0.75      0.83        20

    accuracy                           0.86        42
   macro avg       0.87      0.85      0.85        42
weighted avg       0.87      0.86      0.86        42


Evaluation Metrics for Tuned Model (Best Params):
Accuracy:  0.8095
Precision: 0.9286
Recall:    0.6500
F1-Score:  0.7647
Confusion Matrix:
[[21  1]
 [ 7 13]]

Classification Report:
              precision    recall  f1-score   support

           M       0.75      0.95      0.84        22
           R       0.93      0.65      0.76        20

    accuracy                           0.81        42
   macro avg       0.84      0.80      0.80        42
weighted